In [ ]:
from Extra.evaluation.test import load_tests


In [ ]:
tests = load_tests()


In [3]:
len(tests)

150

In [4]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [5]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [ ]:
from app.core.config import load_settings
from app.services.ollama_client import OllamaClient
from app.services.qdrant_store import QdrantStore
from app.services.retrieval import Retriever
from Extra.evaluation.eval import evaluate_answer, evaluate_retrieval

settings = load_settings()
ollama = OllamaClient(settings)
qdrant = QdrantStore(settings)
qdrant.ensure_collection()
retriever = Retriever(ollama, qdrant, settings.max_retrieval_results)


In [ ]:
retrieval_result = await evaluate_retrieval(retriever, example)
retrieval_result


RetrievalEval(mrr=0.6666666666666666, ndcg=0.6399069297160626, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [ ]:
answer_result, answer, chunks = await evaluate_answer(retriever, ollama, example)


In [ ]:
answer_result


AnswerEval(feedback="The answer correctly identifies the winner as Maxine in the role of Senior Data Engineer but fails to specify her last name, 'Thompson', which is part of the reference answer. It also introduces a different title ('Insurellm Innovator of the Year') contrasted with the original 'Insurellm Innovator of the Year (IIOTY)', possibly adding confusion. Since it does not fully match the reference answer regarding the name, the response is accurate but incomplete. It remains highly relevant, directly answering the question about the award winner.", accuracy=4.0, completeness=3.0, relevance=5.0)

In [ ]:
print(answer_result.feedback)
print(answer_result.accuracy)
print(answer_result.completeness)
print(answer_result.relevance)


The answer correctly identifies the winner as Maxine in the role of Senior Data Engineer but fails to specify her last name, 'Thompson', which is part of the reference answer. It also introduces a different title ('Insurellm Innovator of the Year') contrasted with the original 'Insurellm Innovator of the Year (IIOTY)', possibly adding confusion. Since it does not fully match the reference answer regarding the name, the response is accurate but incomplete. It remains highly relevant, directly answering the question about the award winner.
4.0
3.0
5.0
